In [1]:
# 2. Robustness of the Word2Vec neighbours

import re
import string
from itertools import combinations
from pathlib import Path

from gensim.models import Word2Vec
from lxml import etree
import nltk
from nltk.corpus import stopwords
import pandas as pd

DATA = Path("../data")   # not included in the repository
TAB = Path("../tab")

SEEDS = [1, 2, 3, 4, 5]
SEMANTIC_NEIGHBOURS = ["ruzie", "gerucht", "rumoer", "krakeel", "tierde"]   # roman forms in Table 2

In [2]:
# Definitions

FIRST_YEAR, LAST_YEAR = 1876, 1939


# Standard Dutch NLTK stop words plus domain-specific forms: administrative formulae,
# archaic and French function words, and abbreviations.
EXTRA_STOPWORDS = {
    "den", "ten", "ter", "eene", "een", "geb", "won", "dewelke", "wij",
    "bureel", "bureele", "heden", "genaamde", "geboren", "dienst",
    "wonende", "agent", "huis", "auto", "plaats", "gedaan", "kwam",
    "rond", "persoon", "weg", "wijk", "oud", "agenten", "bestatigd",
    "eenen", "zijne",
    "staat", "politie", "welke", "vastgesteld", "bevel", "fr", "overgebracht",
    "gebracht", "aldaar", "la", "le", "id", "wegens", "genaamden", "waar",
    "volgende", "zijnde", "et", "mijne", "antwerpen", "des", "tusschen",
    "post", "waarvan", "enz", "woonende", "beroep", "commissaris", "plichten",
    "bon", "du", "hare", "langs", "toe", "achter", "get", "uur", "kwartier",
    "ne", "zijnen",
}


PUNCT = re.compile("[%s]" % re.escape(string.punctuation))


def incident_texts(xml_file):
    """Descriptions of all incidents in one register. Text inside NER tags (names, ages,
    occupations, dates, places) is replaced by '_', so that only the officer's wording remains."""
    root = etree.parse(str(xml_file)).getroot()
    for el in root.iter():
        if el.tag not in {"page", "incident", "p", "marg"}:
            el.text = "_"
    return [" ".join(t.strip() for t in inc.itertext() if t.strip())
            for inc in root.xpath(".//incident")]


def tokenize(text):
    """Lowercase, strip punctuation, keep alphabetic tokens."""
    tokens = (PUNCT.sub("", t.lower()) for t in text.split())
    return [t for t in tokens if t.isalpha() and t != "_"]


def load_corpus(registers_file, xml_dir):
    """Training corpus: one row per incident, with year, district and tokens.

    Incidents with fewer than 10 tokens are dropped; after stop-word removal,
    incidents with fewer than 5 remaining tokens are dropped as well."""
    nltk.download("stopwords", quiet=True)
    stop = set(stopwords.words("dutch")) | EXTRA_STOPWORDS

    registers = pd.read_excel(registers_file)
    registers = registers[registers["year"].between(FIRST_YEAR, LAST_YEAR)]
    rows = [{"year": r.year, "district": r.district, "tokens": tokenize(text)}
            for r in registers.itertuples()
            for text in incident_texts(xml_dir / f"{r.archive_file}_final.xml")]
    corpus = pd.DataFrame(rows)
    corpus = corpus[corpus["tokens"].map(len) >= 10]
    corpus["tokens"] = corpus["tokens"].map(lambda ts: [t for t in ts if t not in stop and len(t) > 1])
    return corpus[corpus["tokens"].map(len) >= 5].reset_index(drop=True)


# Hyperparameters shared by all embedding models (paper, footnote 2)
EMBEDDING_PARAMS = dict(sg=1, vector_size=100, window=10, epochs=10, negative=5, workers=4)

In [3]:
sentences = load_corpus(DATA / "gebeurtenisboeken.xlsx", DATA / "xml")["tokens"].tolist()
runs = {seed: Word2Vec(sentences, min_count=10, seed=seed, **EMBEDDING_PARAMS) for seed in SEEDS}

In [4]:
# Mean pairwise Jaccard overlap of the top 10 and top 20 neighbours of lawaai.

def top(model, n):
    return {w for w, _ in model.wv.most_similar("lawaai", topn=n)}

for n in (10, 20):
    scores = [len(top(runs[a], n) & top(runs[b], n)) / len(top(runs[a], n) | top(runs[b], n))
              for a, b in combinations(SEEDS, 2)]
    print(f"top {n}: mean Jaccard {sum(scores) / len(scores):.2f}")

top 10: mean Jaccard 0.61


top 20: mean Jaccard 0.76


In [5]:
# Rank of the semantically related neighbours of Table 2 in each run.

ranks = pd.DataFrame({
    f"seed {seed}": {w: [x for x, _ in model.wv.most_similar("lawaai", topn=200)].index(w) + 1
                     for w in SEMANTIC_NEIGHBOURS}
    for seed, model in runs.items()
})
ranks["min"], ranks["max"] = ranks.min(axis=1), ranks.max(axis=1)
ranks.to_csv(TAB / "robustness_ranks.csv")
ranks

,seed 1,seed 2,seed 3,seed 4,seed 5,min,max
ruzie,1,6,1,3,1,1,6
gerucht,3,1,17,14,11,1,17
rumoer,22,33,14,20,7,7,33
krakeel,6,7,6,8,8,6,8
tierde,35,27,32,27,59,27,59
